In [2]:
import numpy as np
import pandas as pd

In [3]:
## load the cleaned dataset

df = pd.read_csv("../dataset/cleaned_dataset.csv", index_col = 0)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [4]:
## Dividing into independent and dependent features

X = df.drop('Exited', axis = 1)
y = df['Exited']

In [5]:
## Split the data in training and testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
## Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
## Saving the scaler
import pickle
with open('../models/scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN Implementation

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [9]:
(X_train.shape[1], ) ## tuple

(12,)

In [10]:
## Build our ANN Model

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1], )),      ## first hidden layer... here we also need to tell number of neurons in input layer. This is only for first hidden layer

    Dense(32, activation='relu'),       ## Second hidden layer, ... here we dont need to tell the shape of previous hidden layer

    Dense(1, activation='sigmoid')      ## Output
])

model.summary()

g:\Deep Learning\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
## compile the model

model.compile(optimizer = "adam", loss = "binary_crossentropy",metrics=['accuracy'])

In [12]:
## Set up the Tensorboard

log_dir="../logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir = log_dir, histogram_freq=1)

In [13]:
## Set up Early Stopping

'''
If the validation loss does not improve for a specified number of epochs, such as five, we can stop training instead of continuing for all epochs. This practice ensures that we don't unnecessarily train the model when it has already reached its global minimum.

By utilizing early stopping, we can save resources and time, while also maintaining a model that generalizes well to new data. The key is to continuously monitor the loss value during training and halt the process when improvements are no longer observed
'''

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True) ## Monitor 5 epochs. When there is not change in val_loss for 5 epochs then stop the training




In [ ]:
## Training the Model

history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100, callbacks=[tensorflow_callback, early_stopping_callback]    ## Here you can also provide batch_size. By default the batch size is 32
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7720 - loss: 0.4837 - val_accuracy: 0.8370 - val_loss: 0.3941
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8454 - loss: 0.3806 - val_accuracy: 0.8565 - val_loss: 0.3554
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8595 - loss: 0.3514 - val_accuracy: 0.8595 - val_loss: 0.3466
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8589 - loss: 0.3428 - val_accuracy: 0.8600 - val_loss: 0.3506
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8631 - loss: 0.3375 - val_accuracy: 0.8590 - val_loss: 0.3451
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8595 - loss: 0.3357 - val_accuracy: 0.8630 - val_loss: 0.3428
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8639 - loss: 0.3315 - val_accuracy: 0.8585 - val_loss: 0.3420
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8665 - loss: 0.3295 - val_ac

Here the number 250 shows the number of iterations required in one epoch

In [15]:
## save the model

model.save('../models/ann_classifier.h5')

In [16]:
## Load Tensorboard Extension

%load_ext tensorboard

In [17]:
%tensorboard --logdir ../logs/fit20261008-235829

In [19]:
## Prediction 


## load the encoder and scaler
with open('../models/onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo=pickle.load(file)

with open('../models/label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('../models/scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)


input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

input_df=pd.DataFrame([input_data])


geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])


input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)


input_scaled=scaler.transform(input_df)
prediction=model.predict(input_scaled)


prediction_proba = prediction[0][0]


if prediction_proba > 0.5:
    print('The customer is likely to exit.')
else:
    print('The customer is not likely to exit.')

g:\Deep Learning\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
The customer is not likely to exit.


# Hyperparameter Tuning

Determining the optimal number of Hidden layers and neurons

In [33]:
from scikeras.wrappers import KerasClassifier

In [34]:
## Define a function to create the model and try different parameters

def create_model(neurons=32, layers=1) :

    model = Sequential()

    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1) :
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model



In [36]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "model__neurons": [16],
    "model__layers": [2],
    "batch_size": [10],
    "epochs": [30]
}
model = KerasClassifier(
    model=create_model,
    verbose=0
)
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=1,
    cv=3,
    scoring="accuracy"
)

grid_result = grid.fit(X_train, y_train)

print("Best parameters:", grid_result.best_params_)
print("Best CV accuracy:", grid_result.best_score_)

g:\Deep Learning\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
g:\Deep Learning\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
g:\Deep Learning\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
g:\Deep

Best parameters: {'batch_size': 10, 'epochs': 30, 'model__layers': 2, 'model__neurons': 16}
Best CV accuracy: 0.855624995487717


In [37]:
# Evaluate on test data

best_model = grid_result.best_estimator_

test_loss, test_accuracy = best_model.model_.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 0.33835503458976746
Test Accuracy: 0.8644999861717224
